In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from implementation.answer import answer_question

d:\disaster\Disaster-Management-RAG\disaster_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4544.61it/s]


In [2]:
selected_answer=[]

In [3]:
from agents import Agent, Runner, SQLiteSession, OpenAIChatCompletionsModel, RunConfig
from openai import AsyncOpenAI
from IPython.display import Markdown, display
import json
from pydantic import BaseModel, Field
import asyncio
from tqdm import tqdm
import os
import requests

In [4]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [5]:
openai_api_key=os.getenv("OPENAI_API_KEY")

In [6]:
run_config=RunConfig(tracing={"api_key": openai_api_key})

In [7]:
if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [8]:
class Scoring(BaseModel):
    score: float = Field(description="Score from 0 to 5, in increments of 0.5")
    diagnosis: str = Field(description="1-2 sentences: what specific fact is wrong, and what the reference actually says")
    fix_instruction: str = Field(description="1-2 sentences, imperative, telling the next answer exactly what to correct")

In [9]:
#for memory
class ImprovementRecord(BaseModel):
    iteration:int 
    average_score: float
    current_answer: str

In [10]:
class ALLRECORDS(BaseModel):
            iteration:int
            average_score:float
            accuracy_score:float
            completeness_score:float
            relevance_score:float
            accuracy_diagnosis:str
            completeness_diagnosis:str
            relevance_diagnosis:str
            accuracy_fix_instruction: str
            completeness_fix_instruction: str
            relevance_fix_instruction: str
            current_answer:str

In [11]:
class Memory():
    def __init__(self):
        self.history: list[ImprovementRecord] = [] 
        

    def add(self,iteration:int,average_score: float, current_answer: str):
        self.history.append(
             ImprovementRecord(
                iteration=iteration,
                current_answer=current_answer,
                average_score=average_score
             )
        )


In [12]:
class Memory_for_agent():
    def __init__(self):
        self.history: list[ALLRECORDS] = []

    def add(self,iteration:int, average_score:float, accuracy_score:float, completeness_score:float, relevance_score:float, accuracy_diagnosis:str, completeness_diagnosis:str, relevance_diagnosis:str, accuracy_fix_instruction: str, completeness_fix_instruction: str, relevance_fix_instruction: str, current_answer:str):
        self.history.append(
            ALLRECORDS(
                iteration=iteration,
                average_score=average_score,
                accuracy_score=accuracy_score, 
                completeness_score=completeness_score, 
                relevance_score=relevance_score, 
                accuracy_diagnosis=accuracy_diagnosis, 
                completeness_diagnosis=completeness_diagnosis, 
                relevance_diagnosis=relevance_diagnosis, 
                accuracy_fix_instruction=accuracy_fix_instruction, 
                completeness_fix_instruction=completeness_fix_instruction, 
                relevance_fix_instruction=relevance_fix_instruction,
                current_answer=current_answer
            )
        )  

In [13]:
model="llama3.1"
base_url = "http://localhost:11434/v1"

In [14]:
ollama_client=AsyncOpenAI(base_url=base_url, api_key="llama")
llama_model=OpenAIChatCompletionsModel(model=model, openai_client=ollama_client)

In [15]:
accuracy_instruction = """You are a ruthless, skeptical fact-checker for RAG answers.
Your default assumption is that the candidate answer contains errors. You must find
evidence to justify giving credit — credit is never assumed.

You will get: a QUESTION, a REFERENCE ANSWER (ground truth), and a CANDIDATE ANSWER.

Your ONLY job: check if the CANDIDATE contradicts or misstates facts found in the REFERENCE.

## Strictness Rules
- Do NOT give the benefit of the doubt. If a statement in the candidate is vague, imprecise,
  or could be read as technically correct only under a generous interpretation, treat it as
  an error, not a pass.
- Before giving a score of 4.5 or 5.0, you must be able to state that you checked every
  sentence in the candidate against the reference and found zero contradictions. If you
  cannot make that claim, the score must be 4.0 or lower.
- A "minor issue" is not "no issue." Do not let minor issues slide into a perfect score
  because the overall answer "sounds right."
- When uncertain whether something counts as an error, resolve the uncertainty by scoring
  LOWER, not higher.
- Do NOT judge missing content — that is someone else's job. Only judge what the candidate
  actually states and got wrong.
- Do not use your own outside knowledge — only compare candidate vs reference.

## Calibration Examples (what NOT to over-score)
- A candidate that is broadly correct but changes one number, date, or condition from the
  reference → 3.0 or lower, NOT 4.0+. Changing a specific fact is a clear error, not minor.
- A candidate that states something the reference presents as conditional (e.g. "in some
  cases") as if it were universal/absolute → this is a misstatement, score 3.5 or lower.
- A candidate that is vaguer than the reference but not technically wrong → this may still
  be a 4.0-4.5 if truly no contradiction exists, but check carefully; vagueness that changes
  meaning IS an error.

Score (0-5, in increments of 0.5, e.g. 3.0, 3.5, 4.0):
5.0 = verified zero contradictions across the entire candidate.
4.5 = one extremely trivial imprecision with no change in meaning.
4.0 = one minor factual issue that a careful reader would notice.
3.5 = one minor issue plus a small imprecision elsewhere.
3.0 = one clear factual error that changes meaning.
2.5 = one clear error plus additional minor issues.
2.0 = multiple incorrect statements.
1.5 = multiple significant errors, some correct content remains.
1.0 = mostly incorrect.
0.5 = almost entirely incorrect, only a fragment is right.
0.0 = completely wrong.

## Self-Checks (do both before answering)
1. If your diagnosis contains words like "missing," "fails to mention," "does not include,"
   or "lacks" — that is NOT an accuracy issue. Delete it; only describe things the
   candidate said that are WRONG.
2. If you are about to give 4.5 or 5.0, re-read the candidate one more time sentence by
   sentence against the reference. If you find even one questionable statement, lower
   the score.

IMPORTANT: The score MUST be a number in increments of 0.5 only (e.g. 3.0, 3.5, 4.0,
4.5, 5.0). Never output values like 3.3, 3.7, or other non-half-point decimals.

Output ONLY this JSON, nothing else:
{
  "score": <float, 0-5, in 0.5 increments>,
  "diagnosis": "<1-2 sentences: what specific fact is wrong, and what the reference actually says>",
  "fix_instruction": "<1-2 sentences, imperative, telling the next answer exactly what to correct>"
}
"""


completeness_instruction = """You are a ruthless, skeptical completeness checker for RAG
answers. Your default assumption is that the candidate is missing something. You must
find evidence of full coverage to justify a high score — credit is never assumed.

You will get: a QUESTION, a REFERENCE ANSWER (ground truth), and a CANDIDATE ANSWER.

Your ONLY job: check if the CANDIDATE covers all the major points present in the REFERENCE.

## Strictness Rules
- Go through the REFERENCE ANSWER point by point. For each distinct point, topic, step,
  or claim in the reference, explicitly check whether the candidate covers it with
  comparable specificity — not just a vague gesture at the same general area.
- A candidate that mentions a topic only in passing, without the specifics the reference
  gives, counts as NOT covering that point. Do not credit shallow mentions as full coverage.
- Before giving a score of 4.5 or 5.0, you must be able to state that every distinct point
  in the reference has a clear counterpart in the candidate with comparable depth. If you
  cannot make that claim, the score must be 4.0 or lower.
- When uncertain whether something counts as covered, resolve the uncertainty by scoring
  LOWER, not higher.
- Do NOT judge whether stated facts are correct or wrong — that is someone else's job.
- Do not use your own outside knowledge — only compare candidate vs reference.

## Calibration Examples (what NOT to over-score)
- Reference lists 5 distinct steps/items, candidate covers 4 with detail and gestures at
  the 5th in one vague sentence → treat the 5th as NOT covered. Score 3.0-3.5, not 4.5.
- Candidate covers all topics the reference mentions but with noticeably less depth or
  specificity throughout → 3.5-4.0 at most, not 4.5-5.0.
- Reference gives a specific example, number, or named item; candidate only speaks in
  generalities without that specific → this is under-coverage, not full coverage.

Score (0-5, in increments of 0.5, e.g. 3.0, 3.5, 4.0):
5.0 = every point in the reference has a clear, comparably detailed counterpart.
4.5 = all points covered, only one trivial detail slightly thinner.
4.0 = covers all major points but noticeably less depth throughout.
3.5 = covers most major points, missing one minor point or specific.
3.0 = missing one major point entirely.
2.5 = missing one major point and shallow elsewhere.
2.0 = missing multiple major points.
1.5 = missing most major points, only fragments covered.
1.0 = covers very little of the reference.
0.5 = barely touches the question.
0.0 = does not address the core question at all.

## Self-Check (do before answering)
If you are about to give 4.5 or 5.0, list every distinct point in the reference mentally
and confirm each one has a specific counterpart in the candidate. If any point is only
vaguely gestured at rather than actually covered, lower the score.

IMPORTANT: The score MUST be a number in increments of 0.5 only (e.g. 3.0, 3.5, 4.0,
4.5, 5.0). Never output values like 3.3, 3.7, or other non-half-point decimals.

Output ONLY this JSON, nothing else:
{
  "score": <float, 0-5, in 0.5 increments>,
  "diagnosis": "<1-2 sentences: name the specific missing topic(s) from the reference>",
  "fix_instruction": "<1-2 sentences, imperative, telling the next answer exactly what to add>"
}
"""


relevance_instruction = """You are a ruthless, skeptical on-topic checker for RAG answers.
Your default assumption is that some content in the candidate is padding, generic, or
off-topic. You must find evidence that every sentence earns its place to justify a
high score — credit is never assumed.

You will get: a QUESTION and a CANDIDATE ANSWER.

Your ONLY job: check if the CANDIDATE stays focused on what the question actually asked.

## Strictness Rules
- Go through the candidate sentence by sentence. For each sentence, ask: does this
  directly answer what was asked, or could it be removed without losing any real answer
  to the question? If it could be removed, it is padding.
- Generic setup sentences, restated questions, filler transitions ("It's important to
  note that..."), and broad statements not specific to the question all count as padding.
- Before giving a score of 4.5 or 5.0, you must be able to state that you checked every
  sentence and found no padding, tangents, or generic filler. If you cannot make that
  claim, the score must be 4.0 or lower.
- When uncertain whether something is padding, resolve the uncertainty by scoring
  LOWER, not higher.
- State definitively whether content is on-topic or off-topic. No hedging language like
  "somewhat" or "a bit" in your diagnosis.
- Do not judge factual correctness or completeness — only focus/relevance.

## Calibration Examples (what NOT to over-score)
- Candidate opens with a generic restatement of the question or topic before answering
  → this is padding, cap the score at 4.0.
- Candidate includes a true but tangential fact not asked about (e.g. history or context
  not requested) → this is off-topic content, score 3.0-3.5 depending on how much of the
  answer it takes up.
- Candidate is otherwise focused but ends with generic closing filler ("In conclusion,
  it's important to always be prepared") → this is padding, cap the score at 4.0-4.5.

Score (0-5, in increments of 0.5, e.g. 3.0, 3.5, 4.0):
5.0 = every sentence directly answers the question; zero padding.
4.5 = one trivial aside, does not read as padding.
4.0 = minor tangent or filler sentence, doesn't distract much.
3.5 = small amount of off-topic or generic content present.
3.0 = noticeable off-topic content mixed in with relevant content.
2.5 = roughly half the answer is padding or off-topic.
2.0 = largely padded with irrelevant or generic content.
1.5 = only small fragments are actually on-topic.
1.0 = mostly off-topic; only incidentally touches the question.
0.5 = barely touches the question at all.
0.0 = does not address the question at all.

## Self-Check (do before answering)
If you are about to give 4.5 or 5.0, re-read the candidate sentence by sentence and ask
of each one: "does this specifically answer the question, or is it generic/filler?"
If any sentence fails that test, lower the score.

IMPORTANT: The score MUST be a number in increments of 0.5 only (e.g. 3.0, 3.5, 4.0,
4.5, 5.0). Never output values like 3.3, 3.7, or other non-half-point decimals.

Output ONLY this JSON, nothing else:
{
  "score": <float, 0-5, in 0.5 increments>,
  "diagnosis": "<1-2 sentences: name the specific off-topic or padding content, if any>",
  "fix_instruction": "<1-2 sentences, imperative, telling the next answer what to remove or refocus on>"
}
"""

In [16]:
def query_form(question,reference_answer,candidate_answer):
    query=f"""Question: {question}\n\nReference Answer:\n\n{reference_answer}\n\nAnswer to be evaluated:\n\n{candidate_answer}"""
    return query

In [17]:

rewrite_instruction = """You are an expert answer-revision assistant for a RAG system.

You will be given:
1. RETRIEVED CHUNKS — the source documents/context the answer should be grounded in.
   This is your ONLY source for any new facts, details, or corrections you add.
2. QUESTION 
2. CANDIDATE ANSWER — a previous answer that was evaluated and found to have issues.
3. EVALUATION FEEDBACK — three sets of scores and fix instructions (Accuracy,
   Completeness, Relevance) describing exactly what is wrong with the CANDIDATE ANSWER.

Your task: rewrite the CANDIDATE ANSWER into an improved answer that resolves every issue
named in the EVALUATION FEEDBACK, using ONLY information found in the RETRIEVED CHUNKS,
while keeping everything that was already correct and relevant.

## Rules
- Treat each "fix_instruction" as a mandatory task — it tells you WHAT to fix. Use the
  RETRIEVED CHUNKS to find the actual correct content to fix it WITH.
- If Accuracy feedback flags something as factually wrong, find the correct fact in the
  RETRIEVED CHUNKS and correct it. If the chunks do not contain enough information to
  confidently fix it, remove the incorrect claim rather than guessing.
- If Completeness feedback flags something as missing, search the RETRIEVED CHUNKS for
  that information and add it, in your own words. If the chunks do not contain the
  specific detail the feedback mentions, do not invent it — add only what the chunks
  actually support, or omit it if it truly is not present in the chunks.
- If Relevance feedback flags something as off-topic or padding, remove or shorten it.
- Do NOT throw out the candidate answer and start from scratch. Preserve sentences and
  structure that were not flagged as problems. Only change what the feedback names.
- Never state a fact, statistic, name, or step that is not present in the RETRIEVED
  CHUNKS. Do not use outside/prior knowledge to fill gaps.
- Do NOT copy chunk text verbatim — paraphrase in your own words while staying factually
  aligned with the chunks.
- Do NOT mention the evaluation, scores, feedback, chunks, or that this is a revision.
  Output only the improved answer itself, as if answering fresh.
- Do not pad the answer with restated content to appear more thorough than necessary.

Output:
Return ONLY the revised answer text. No preamble, no headers, no explanation of changes.

"""

In [18]:
def user_prompt(result1, result2, result3, chunks, candidate_answer,question):
    final_prompt="""##Inputs:"""
    accuracy_score=result1["score"]
    accuracy_diagnosis=result1["diagnosis"]
    accuracy_fix_instruction=result1["fix_instruction"]
    completeness_score=result2["score"]
    completeness_diagnosis=result2["diagnosis"]
    completeness_fix_instruction=result2["fix_instruction"]
    relevance_score=result3["score"]
    relevance_diagnosis=result3["diagnosis"]
    relevance_fix_instruction=result3["fix_instruction"]

    add_on=f"""RETRIEVED CHUNKS:
{chunks}

QUESTION:
{question}

CANDIDATE ANSWER:
{candidate_answer}

EVALUATION FEEDBACK:
Accuracy (score: {accuracy_score}/5) — {accuracy_diagnosis} FIX: {accuracy_fix_instruction}
Completeness (score: {completeness_score}/5) — {completeness_diagnosis} FIX: {completeness_fix_instruction}
Relevance (score: {relevance_score}/5) — {relevance_diagnosis} FIX: {relevance_fix_instruction}"""
    final_prompt+= "\n"
    final_prompt+=candidate_answer 
    final_prompt+=add_on
    return final_prompt

In [19]:
agent_acc=Agent(name="Accuracy evaluator", instructions=accuracy_instruction, model=llama_model, output_type=Scoring)
agent_com=Agent(name="compeletion evaluator", instructions=completeness_instruction, model=llama_model, output_type=Scoring)
agent_rel=Agent(name="relevance evaluator", instructions=relevance_instruction, model=llama_model, output_type=Scoring)
agent_rewriter=Agent(name="Answer rewriter", instructions=rewrite_instruction, model=llama_model)

In [20]:
async def running_para(query):
    results = await asyncio.gather(
    Runner.run(agent_acc, query,run_config=run_config),
    Runner.run(agent_com, query,run_config=run_config),
    Runner.run(agent_rel, query,run_config=run_config)
    )
    output=[json.loads(result.final_output.model_dump_json()) for result in results]  
    return output
      

In [21]:
iteration=5

In [22]:
#push notification about the final accuaracy
def push(message):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [23]:
def select_the_best(memory):
    best_record = max(memory.history, key=lambda r: r.average_score)
    return best_record.model_dump_json()

In [24]:
async def coach(question,reference_answer,iteration, memory, session):
    candidate_answer, chunks=answer_question(question)
    query=query_form(question, reference_answer, candidate_answer)
    results= await running_para(query)
    count=0
    while any(result["score"] < 4.5 for result in results) and count < iteration:
        average_score=(results[0]["score"]+results[1]["score"]+results[2]["score"])/3
        memory.add(
            iteration=count,
            average_score=average_score,
            current_answer=candidate_answer
        )
        prompt=user_prompt(results[0], results[1], results[2], chunks, candidate_answer, question)
        potential_answer=await Runner.run(agent_rewriter, prompt, session=session, run_config=run_config)
        count+=1
        candidate_answer=potential_answer.final_output
        query=query_form(question, reference_answer, candidate_answer)
        results=await running_para(query)     

In [ ]:
with open(r"D:\disaster\Disaster-Management-RAG\evaluation\tests.jsonl", "r", encoding="utf-8") as f:
    with open(r"D:\disaster\Disaster-Management-RAG\selected_answer.txt", "w", encoding="utf-8") as t:
        content=f.readlines()
        session = SQLiteSession("12346")
        for line in tqdm(content):
            line= json.loads(line)
            question=line["question"] 
            reference_answer=line["reference_answer"]
            memory=Memory()
            await coach(question,reference_answer,iteration, memory, session)
            record=select_the_best(memory)
            selected_answer.append(json.loads(record))
            t.write(json.dumps(selected_answer)+"\n")
            push(json.dumps(record))
            print(record.current_answer)
            
        

  0%|          | 0/150 [00:00<?, ?it/s][non-fatal] Tracing: request failed: [Errno 11002] getaddrinfo failed
[non-fatal] Tracing: request failed: [Errno 11002] getaddrinfo failed
[non-fatal] Tracing: request failed: [Errno 11002] getaddrinfo failed
[non-fatal] Tracing: max retries reached, giving up on this batch.
  0%|          | 0/150 [05:15<?, ?it/s]


In [ ]:

async def improvement():
    selected_answer = []
    with open(r"D:\disaster\Disaster-Management-RAG\evaluation\tests.jsonl", "r", encoding="utf-8") as f:
        content = f.readlines()
        session = SQLiteSession("12346")
        for line in tqdm(content):
            line = json.loads(line)
            question = line["question"]
            reference_answer = line["reference_answer"]
            memory = Memory()
            await coach(
                question,
                reference_answer,
                iteration,
                memory,
                session,
                )
            record = select_the_best(memory)
            selected_answer.append(json.loads(record))
            push(json.dumps(record))
            temp=json.loads(record)
            return temp["current_answer"]


